# Healthier people. Better insurance?

**A guided investigation · Chapter 1 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-nhis_clean.ipynb)

> **The question**  
> Does having insurance explain why insured people report better health?

A health survey can reveal a striking gap without telling us what caused it. The 2009 National Health Interview Survey asked people about their health, insurance, and household circumstances. This teaching extract selects married adults ages 26–59 in households with an employed adult; it is not a picture of every American. Each row records one person, including a self-reported health score, schooling, and family income. Insurance was not randomly assigned. People with coverage may also have jobs, resources, or other advantages that shape their health. Start by comparing health distributions, then look at who belongs to each insurance group. The question becomes more interesting as the comparison becomes less simple: if the groups differ in several ways, how could we separate the contribution of insurance from those other differences?

### By the end, you will be able to…

- Compare health distributions, then look for differences that make a causal interpretation difficult.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

If insured people report better health, what are two explanations besides an effect of insurance?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Person in a selected sample of married adults ages 26–59 in households with an employed adult.  
**Stored size:** 18,790 rows × 10 columns.

| Column | What it records |
|---|---|
| `health` | Self-reported health score: 1 poor, 2 fair, 3 good, 4 very good, 5 excellent. |
| `insurance` | Insurance coverage indicator: 1 insured, 0 uninsured. |
| `education` | Years of education. |
| `family_income` | Family income in source dollars; NHIS is constructed from income categories, RAND baseline uses source inflation-adjusted income. |
| `weight` | Source person survey weight for NHIS; sample-line analysis weight for child-labor cells. Not a count of stored rows. |

Health is an ordered response from 1 (poor) to 5 (excellent), not a clinical measurement. This selected household sample is not the entire NHIS. Initial summaries give each row equal weight.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch1/nhis_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['health', 'insurance', 'education', 'family_income', 'weight']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['health', 'insurance', 'education', 'family_income', 'weight']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'age': 'Age (years)',
 'census': 'Census year',
 'education': 'Education (years)',
 'group': 'State group',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state'}

## Visual investigation 1 · A health gap is a distribution

**Predict:** Are the two groups spread across health categories in the same way?

A count answers how many; a percentage makes groups of different sizes comparable. groupby collects rows sharing the same labels. size counts those rows. Dividing by each group total turns counts into percentages.

In [ ]:
health_counts = df.groupby(["insurance", "health"]).size().reset_index(name="people")
health_counts["percent"] = 100 * health_counts["people"] / health_counts.groupby("insurance")["people"].transform("sum")
health_counts["coverage"] = health_counts["insurance"].map({0: "Uninsured", 1: "Insured"})
fig = px.bar(health_counts, x="health", y="percent", color="coverage", barmode="group", labels={"health": "Health score: 1 poor → 5 excellent", "percent": "People within coverage group (%)", "coverage": "Insurance"}, title="How people describe their health")
fig.show()

**Explore the chart:** Hover over health scores 1 and 5. Compare the percentages, not just the color. Toggle a group in the legend.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Describe where each distribution is concentrated. A health gap is an association; the chart does not tell us why coverage and health differ.

</details>

## Visual investigation 2 · Who has insurance?

**Predict:** Do the groups also differ in education?

A box plot summarizes a distribution: the box covers its middle half and the line inside marks its median, the middle value. We hide individual points to keep this large sample readable.

In [ ]:
people = df.copy()
people["coverage"] = people["insurance"].map({0: "Uninsured", 1: "Insured"})
fig = px.box(people, x="coverage", y="education", color="coverage", points=False, labels={"coverage": "Insurance", "education": "Years of education"}, title="Schooling differs across insurance groups")
fig.show()

**Explore the chart:** Hover over each box to inspect its median and quartiles. Would the same health comparison be persuasive if schooling differed substantially?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Education is one observed difference that could help explain the health gap. It is not a complete list of confounders, and the box plot does not estimate their contributions.

</details>

## Visual investigation 3 · Resources behind the comparison

**Predict:** How different are typical family incomes?

The median is less sensitive to very high incomes than the mean. Here we calculate one median per coverage group. Family income describes a household resource even though the stored rows are people.

In [ ]:
income = people.groupby("coverage", as_index=False)["family_income"].median()
fig = px.bar(income, x="coverage", y="family_income", color="coverage", labels={"coverage": "Insurance", "family_income": "Median family income (source dollars)"}, title="Another difference to explain")
fig.show()

**Explore the chart:** Hover to read the income values. Name a way family resources might affect both insurance and health.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The insured–uninsured comparison bundles insurance with other circumstances. These data invite a stronger research design; they do not isolate an insurance effect.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

In the education box plot, change education to age and update the axis label. Predict which group will be older.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Make a chart of mean health by coverage group. Then explain why its simplicity could be misleading.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Start with the people table, which already has readable coverage labels.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Use people.groupby("coverage", as_index=False)["health"].mean(), then px.bar.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>means = people.groupby(&quot;coverage&quot;, as_index=False)[&quot;health&quot;].mean()
fig = px.bar(means, x=&quot;coverage&quot;, y=&quot;health&quot;, color=&quot;coverage&quot;, labels={&quot;coverage&quot;: &quot;Insurance&quot;, &quot;health&quot;: &quot;Mean health score (1–5)&quot;}, title=&quot;A useful summary, not a causal effect&quot;)
fig.show()</code></pre>

A mean compresses an ordered health distribution into one number. The earlier schooling and income charts show why that difference cannot be attributed to insurance alone.

Copy the code into the starter cell to run it.

</details>

## Optional extension · do weights change the picture?

Person survey weights give more influence to respondents representing more people. This remains a descriptive comparison of the selected extract, not a survey-design variance estimate.

A **weighted mean** multiplies each value by its weight, adds those products, and divides by the sum of weights. The code below drops rows missing any required value and keeps positive weights. Each group gets its own denominator. This is a descriptive calculation, not an uncertainty estimate or a causal adjustment.

Predict whether weighting will change the comparison. Then compare the two summaries and explain which units receive more influence.

In [ ]:
weighted_data = df.dropna(subset=["health", "insurance", "weight"]).copy()
weighted_data = weighted_data[weighted_data["weight"] > 0]
weighted_data["product"] = weighted_data["health"] * weighted_data["weight"]
totals = weighted_data.groupby("insurance")[["product", "weight"]].sum()
weighted_mean = totals["product"] / totals["weight"]
unweighted_mean = weighted_data.groupby("insurance")["health"].mean()
comparison = pd.DataFrame({"Equal row weights": unweighted_mean, "Source weights": weighted_mean})
display(comparison.round(3))

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**What would random assignment of insurance change about the comparison you just made?**

[Continue with Chapter 1: Randomized Trials](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/01-randomized-trials.html#does-health-insurance-improve-health) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch1-nhis-clean-csv)

### Source and preparation notes

[National Center for Health Statistics, 2009 NHIS data release](https://archive.cdc.gov/www_cdc_gov/nchs/nhis/nhis_2009_data_release.htm)

`code/python/ch1/prepare_data.py` selects married adults, positive person weight, nonmissing own/spouse insurance, one woman per household, ages 26–59, employed household, and households with more than one sampled person. It recodes Stata categories, creates household size/spouse insurance, and writes 10 teaching columns. Survey weight is retained as `weight`; displayed profiles remain unweighted.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.